# Web scraping

Le note di questa lezione affrontano il problema di estrazione di dati da pagine web mostrando 
una serie di approcci, da quello basato sulla libreria standard di Python ad alcuni basati su librerie esterne.

Sono organizzate come segue:

* estrarre i dati di contatto dal [Chi e dove](https://www.unimi.it/it/chi-e-dove) con [urllib](https://docs.python.org/3/library/urllib.html) e [re](https://docs.python.org/3/library/re.html);
* realizazzione di un semplice pacchetto Python col *package manager* moderno [uv](https://docs.astral.sh/uv/);
* realizzazione di un tool da linea di comando per l'estrazione dei dati di contatto dal [Chi e dove](https://www.unimi.it/it/chi-e-dove) con [requests](https://docs.python-requests.org/en/latest/) e [BeautifulSoup](https://beautiful-soup-4.readthedocs.io/);
* esempio [Selenium](https://www.selenium.dev/) per l'estrazione delle [liste appelli ed iscritti
](https://work.unimi.it/boDocenti/ListaAppelliEIscritti).

## Batteries included: `urllib` e `re`

In questa sezione vedremo come usare le espressioni regolari (secondo quanto discusso nella precedente lezione) per estrarre i dati di contatto dei docenti dal sito [Chi e dove](https://www.unimi.it/it/chi-e-dove) usando la libreria standard di Python `urllib` per scaricare il contenuto della pagina come discusso in [HOWTO Fetch Internet Resources Using The urllib Package](https://docs.python.org/3/howto/urllib2.html).

In [18]:
from html import unescape
from re import finditer, search, DOTALL
from urllib.request import urlopen, Request
from urllib.parse import urlencode

Per prima cosa, scegliamo un cognome "Cesa" e procuriamoci con una *get* l'elenco di nomi di docenti il cui [cognome contiene "Cesa"](https://www.unimi.it/it/chi-e-dove?cognome=cesa).

In [19]:
SEARCH_URL = 'https://www.unimi.it/it/chi-e-dove'

cognome = 'cesa'

url = SEARCH_URL + '?' + urlencode({
  'cognome': cognome,
  'nome': '',
  'ur_tipi_ruoli_target_id': 'All',
})

risultato = urlopen(
  Request(url, headers = {'User-Agent': 'Mozilla/5.0'})
).read().decode('utf-8')

Per farci una idea di come cercare, vediamo dove è contenuto il cognome

In [20]:
for line in risultato.splitlines():
  if cognome in line:
      print(line)

    <script type="application/json" data-drupal-selector="drupal-settings-json">{"path":{"baseUrl":"\/","pathPrefix":"it\/","currentPath":"node\/7663","currentPathIsAdmin":false,"isFront":false,"currentLanguage":"it","currentQuery":{"cognome":"cesa","nome":"","ur_tipi_ruoli_target_id":"All"}},"pluralDelimiter":"\u0003","suppressDeprecationErrors":true,"ajaxPageState":{"libraries":"eJyVVO1y4yAMfCHHPJJHGBmrBcQgSJo-_SkxcZyb3qT3B2tXK9DowxZrxTLhV2ZBNy0UFIrxmLBAGGyA76uxxCN8wNdgmavUAtns1pQLTpSoHpyZM5-xHJjKHCrl4RAGBbwaqzzvOj3JfyjzaebQYpJhXjXhZLbPaHF5MK60DGHcUCenQPYhnUWDmT8JxfTvh5RxduknWuo14B7Q62Ne4e71YCBBuFaanyG-gje3Q5mCj_QcQWC_VXXhVOGCwhHNwR4vaG9QRlsgOXmvm5tUju91rnHlhO-FBX0LUN4LZYWSA_m1_lL7fzcLB3K_0P5OVldKg2f2Aae9OX_hrTMvZBzIYrFZl6X-OLMndbxo8qleeJ_YAFdudXIk8207rkZboL7hMN99OJ7M2FJuNpCs6Ibm-TzxYnxgq7t5hzm8wNJsoRl2TpcuYmpG0ykgT7xCtK14nd2d4mWZIZ1ByzhDQC3R7pJPSpoJRRrup4Gsr6zU3IPYy9FxH_6O7mXv9pbZ6bZXlHwns9ZG_0IFhb6xc4_XO7wATRF0vHk4Q7EgOEXULdKnYuQ0nAkvYu7n1rsjEdm1gH8AYBTxEw","theme":"unimi","theme_t

Questo suggerisce l'uso di una espressione regolare basata su quel che segue `<a href="/it/ugov/person/`

In [21]:
persone = []
for match in finditer(r'<a href="/it/ugov/person/([^"]+)"', risultato):
    slug = match.group(1)
    if slug not in persone:
        persone.append(slug)

persone

['nicolo-cesa-bianchi',
 'roberta-cesana',
 'matteo-cesari',
 'nicoletta-cesari',
 'valentina-cesari',
 'franco-cesaro']

In [22]:
DETAIL_URL = 'https://www.unimi.it/it/ugov/person/'

persona = 'nicolo-cesa-bianchi'

dettaglio = urlopen(Request(DETAIL_URL + persona, headers = {'User-Agent': 'Mozilla/5.0'})).read().decode('utf-8')

Analizzando il dettaglio, una serie di espressioni regolari consentono di identificare alcune parti di interesse.

In [23]:
# unescape è necessario per convertire i caratteri HTML entities in caratteri normali
# dovuti ad esempio dalle accentate

unescape(search(r'<h1 class="page-header">(.+?)</h1>', dettaglio).group(1))

"Cesa Bianchi Nicolo' Antonio"

In [24]:
search(r'<p class="ugov-indirizzo">(.+?)</p>', dettaglio).group(1)

'Via Celoria, 18'

Più difficile il caso del telefono che compare in una contesto del tipo

```html
<div ...>
<div class="... phone ...">Numero di telefono dell'ufficio</div>
<div ...">02503...</div>
</div>
```

che può essere catturata dal seguente pattern (che cattura due `div` in sequenza):

In [25]:
pattern = r'\s*'.join([
  r'<div[^>]*\bphone\b[^>]*>.*?</div>', 
  r'<div[^>]*>(.*?)</div>']
)

search(pattern, dettaglio, flags = DOTALL).group(1).strip()

'02503 16280'

L'email è *offuscata* usando una tecnica ispirata a una [libreria di Cloudflare](developers.cloudflare.com/waf/tools/scrape-shield/email-address-obfuscation/).

In [26]:
cf_email = search(r'<span.*?data-cfemail="([^"]*)"', dettaglio).group(1)

cf_email

'e08e89838f8c8fce83859381cd8289818e838889a0958e898d89ce8994'

Le funzioni di codifica e decodifica sono:

In [27]:
def encode_cfemail(email, key = 1):
  result = [key] + [b ^ key for b in email.encode('utf-8')]
  return bytearray(result).hex()

In [28]:
def decode_cfemail(encoded):
  key, text = int(encoded[:2], 16), bytes.fromhex(encoded[2:])
  return ''.join(chr(b ^ key) for b in text)

In [29]:
esempio = encode_cfemail('ciao')
decode_cfemail(esempio)

'ciao'

Nel caso delle informazioni nel dettaglio, si ha

In [30]:
decode_cfemail(cf_email)

'nicolo.cesa-bianchi@unimi.it'

Seguendo questi passi, è possibile assemblare uno strumento da linea di comando che, dato il cognome, emetta tutti i dati raccolti. Un esempio di implementazione è disponibile nel pacchetto [chiedove-unimi](https://github.com/mapio/chiedove-unimi).

In genere l'uso diretto delle funzioni della libreria standard rende però sia la parte di rete che quella di estrazione dei dati più complessa e meno robusta. Per questo motivo, è spesso preferibile affidarsi a librerie esterne.

## Creare un pacchetto Python con `uv`

Prima di dedicarci alla riscrittura del precedente strumento usando librerie esterne, vediamo come creare un pacchetto Python usando il *package manager* moderno [uv](https://docs.astral.sh/uv/).

Una volta [installato](https://docs.astral.sh/uv/getting-started/installation/) possiamo procedere con la creazione di un paccetto col comando

In [35]:
! uv init --package esempio

Initialized project `esempio` at `/chome/santini/Activities/Teaching/Courses/pytab/lezioni/esempio`


Questo comando crea una cartella `esempio` con all'interno una struttura di base per un pacchetto Python. La cartella `src` contiene il codice del pacchetto, mentre `pyproject.toml` contiene le informazioni sul pacchetto e le sue dipendenze.

In [40]:
# questo comando è eseguibile su GNU/Linux, chi usa Window dovrà trovare l'equivalente...

! tree esempio

esempio
├── pyproject.toml
├── README.md
└── src
    └── esempio
        └── __init__.py

3 directories, 3 files


In [41]:
# questo comando è eseguibile su GNU/Linux, chi usa Window dovrà trovare l'equivalente...

! cat esempio/pyproject.toml

[project]
name = "esempio"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
authors = [
    { name = "Massimo Santini", email = "santini@di.unimi.it" }
]
requires-python = ">=3.12"
dependencies = []

[project.scripts]
esempio = "esempio:main"

[build-system]
requires = ["uv_build>=0.11.14,<0.12.0"]
build-backend = "uv_build"


Per eseguire il comando (che avrà nome uguale a quello del pacchetto) si può usare `uv run` 

In [43]:
! cd esempio && uv run esempio

Hello from esempio!


Una volta che il codice è ritenuto pronto, si può creare un pacchetto installabile con il comando `uv build`

In [52]:
! cd esempio && uv build

Building source distribution (uv build backend)...
Successfully built dist/esempio-0.1.0.tar.gz
Successfully built dist/esempio-0.1.0-py3-none-any.whl


Questo crea una *wheel* ossia una [distribuzione binaria](packaging.python.org/en/latest/specifications/binary-distribution-format/) che può essere distribuita e quindi installata (con `pip`, `uv` o altri strumenti analoghi).

In [54]:
! ls esempio/dist/*.whl

esempio/dist/esempio-0.1.0-py3-none-any.whl


## Riscrivere `chiedove-unimi` con `uv` e librerie esterne

Siamo pronti per riscrivere lo strumento di estrazione dei dati di contatto dal [Chi e dove](https://www.unimi.it/it/chi-e-dove) usando `uv` e le librerie esterne `requests` e `BeautifulSoup`.

In [55]:
! uv init --package chiedove-unimi

error: Project is already initialized in `/chome/santini/Activities/Teaching/Courses/pytab/lezioni/chiedove-unimi` (`pyproject.toml` file exists)


In [56]:
! cd chiedove-unimi && uv add requests beautifulsoup4 

Resolved 9 packages in 0.82ms
Checked 9 packages in 0.16ms


Vediamo come riscrivere lo scaricamento del dettaglio e l'estrazione del numero di telefono usando `requests` e `BeautifulSoup`, il resto del codice è nella directory [chiedove-unimi-santini](chiedove-unimi-santini)

In [57]:
DETAIL_URL = 'https://www.unimi.it/it/ugov/person/'

persona = 'nicolo-cesa-bianchi'

dettaglio = urlopen(Request(DETAIL_URL + persona, headers = {'User-Agent': 'Mozilla/5.0'})).read().decode('utf-8')

pattern = r'\s*'.join([
  r'<div[^>]*\bphone\b[^>]*>.*?</div>', 
  r'<div[^>]*>(.*?)</div>']
)

search(pattern, dettaglio, flags = DOTALL).group(1).strip()

'02503 16280'

## L'uso di Selenium per l'estrazione di dati da pagine dinamiche

Se il contenuto della pagina è dinamico, ossia viene generato da codice JavaScript, allora è necessario usare uno strumento che consenta di eseguire quel codice e quindi di estrarre i dati. Uno strumento molto popolare per questo scopo è [Selenium](https://www.selenium.dev/).

Sviluppare un esempio "passo passo" è un po' complesso, ma è possibile trovare un esempio completo di estrazione delle [liste appelli ed iscritti](https://work.unimi.it/boDocenti/ListaAppelliEIscritti) con Selenium nel repository [bodocenti-unimi](https://github.com/mapio/bodocenti-unimi)